# headswap_V2 - test: pre-edit donor expression before T4

**Run order:** Cell 1 (setup, restarts the kernel) -> Cell 2 (upload) -> Cell 3 (run).

Cell 1 detects whether this runtime already has ComfyUI installed:
- **Fresh runtime** -> full setup (clone, deps, ComfyUI, Krea2 weights).
- **Existing runtime** -> skips the slow install, just re-syncs the repo to the latest commit on this branch.

Cell 3 tests the new `pre_edit_donor_expression` step (docs/PIPELINE_STATE.md CHECKPOINT-11/12/13): it measures the **target's** actual expression, edits the **donor** photo to match it (pure Krea2 generation, no mask), then runs T4's existing two-step pass (main pass + face_refine) on the edited donor. The cell displays all three stages: the original donor face, the donor after the expression edit, and T4's final result.


In [ ]:
#@title Cell 1 - Setup (detects an existing runtime; only a fresh one gets the full install)
from pathlib import Path
import subprocess, shutil, os, signal, sys
import importlib.metadata as _im

assert Path("/content").exists(), "Open this notebook in Google Colab."

def _import_torch():
    import torch
    torch.cuda.is_available()  # touch a real attribute to force full init
    return torch

try:
    torch = _import_torch()
except AttributeError as exc:
    # Known Colab base-image hiccup, seen on a brand-new runtime's very
    # first `import torch`: the module partially initializes and a later
    # submodule (torch.fx via torch._export.verifier) is missing, raised
    # as "partially initialized module ... most likely due to a circular
    # import". Reinstalling the SAME pinned version (not upgrading, which
    # could pull a build mismatched with Colab's GPU driver) refreshes
    # whatever got corrupted.
    try:
        _torch_ver = _im.version("torch")
    except _im.PackageNotFoundError:
        _torch_ver = None
    pkg = f"torch=={_torch_ver}" if _torch_ver else "torch"
    print(f"torch import broken on this runtime ({exc}); reinstalling {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-deps", "--no-cache-dir", pkg],
                   check=True)
    for _m in list(sys.modules):
        if _m == "torch" or _m.startswith("torch."):
            del sys.modules[_m]
    torch = _import_torch()

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run this cell.")
print(f"GPU {torch.cuda.get_device_name(0)}")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
BRANCH = "simple-full-body-head-swap"

# Fresh runtime: no ComfyUI on disk yet -> needs the full install below.
# Existing runtime (reconnect / re-run): ComfyUI + weights are already on
# disk -> only re-sync the repo, skip the slow scripts/setup_colab.sh.
FRESH_RUNTIME = not Path("/content/ComfyUI/server.py").exists()
if FRESH_RUNTIME:
    print("-> Fresh runtime detected (no ComfyUI on disk) - running full setup.")
else:
    print("-> Existing runtime detected (ComfyUI already on disk) - skipping "
          "the slow install, just re-syncing the repo.")

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"),
      subprocess.getoutput(f"git -C {REPO} log -1 --pretty=%s"))

os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

if FRESH_RUNTIME:
    shutil.rmtree("/content/ComfyUI", ignore_errors=True)
    r = subprocess.run(["bash", "scripts/setup_colab.sh", "--no-drive", "--krea2"],
                       check=False, capture_output=True, text=True)
    print("setup exit:", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("--- stderr ---"); print(r.stderr[-2000:])
        raise SystemExit("setup_colab.sh failed")
else:
    print("ComfyUI + weights already present - skipping scripts/setup_colab.sh")

subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                "--no-deps", "numpy==2.4.6"], check=True)

print("\n✓ Setup complete. Restarting kernel (this is expected)...")
print("   When it comes back, run Cell 2.")
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
#@title Cell 2 - Upload YOUR two images
# Upload the BODY first (the photo you want to keep: pose, clothes, background;
# this is also where the DESIRED expression is measured from), then the FACE
# (the donor whose identity you want transferred in -- its expression will be
# edited by Cell 3 to match the body's before the swap).
import os
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "my_pair"
PAIR.mkdir(parents=True, exist_ok=True)
for old in PAIR.glob("*"):
    old.unlink()

def _grab(role):
    print(f"\n=== Upload the {role} image ===")
    up = files.upload()
    if not up:
        raise SystemExit(f"No {role} image uploaded - re-run this cell.")
    name = next(iter(up))
    dest = PAIR / f"{role}.png"
    Image.open(name).convert("RGB").save(dest)
    os.remove(name)
    im = Image.open(dest)
    print(f"saved {role}: {im.size[0]}x{im.size[1]}px")
    if max(im.size) < 500:
        print(f"   note: small source ({im.size[0]}x{im.size[1]}). The pipeline "
              "upscales to 1024 so generated detail survives, but a larger "
              "original will always look sharper.")
    return im

body_im = _grab("body")
face_im = _grab("face")

print("\n--- BODY (kept: pose / clothing / background / DESIRED expression) ---")
display(body_im)
print("--- FACE (donor: identity; expression will be edited to match BODY) ---")
display(face_im)
print("\n✓ Ready. Run Cell 3.")


In [ ]:
#@title Cell 3 - Run: identity_lora_strength sweep on T4's main pass
SEED = 46  #@param {type:"integer"}

# THE LEVER UNDER TEST (docs/PIPELINE_STATE.md CHECKPOINT-13 calls this "the
# only untested lever", and it is still untested -- the pre-edit detour never
# came back to it). The identity LoRA is trained to transplant the head from
# image 2, and a head includes its expression. This is the global dial on how
# hard it does that. T4's own value is 1.0.
#
# Sweep 1.0 -> 0.7 -> 0.5, one run each, same seed. Judge BOTH axes every
# time: did the expression move, and did the identity survive. The likely
# failure mode -- seen on every other lever in this investigation -- is that
# they move together, i.e. the smile relaxes only as the face stops being the
# donor. If that happens the global dial is the wrong shape of control and
# the answer is per-block or timestep-scheduled LoRA, not a different number.
IDENTITY_LORA_STRENGTH = 1.0  #@param {type:"number"}

# Donor pre-edit: CLOSED as a dead end (CHECKPOINT-14) -- four GPU rounds,
# zero expression movement, and round 4 was verified unconfounded. Left here
# only so the arm can be re-run for reference; leave it False.
RUN_DONOR_PRE_EDIT = False  #@param {type:"boolean"}
USE_CODE_DEFAULTS = True  #@param {type:"boolean"}
PRE_EDIT_DENOISE = 0.45  #@param {type:"number"}
PRE_EDIT_REF_BOOST = 2.0  #@param {type:"number"}
PRE_EDIT_CFG = 4.0  #@param {type:"number"}
PRE_EDIT_DISABLE_LORA = True  #@param {type:"boolean"}

import sys, os, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
os.chdir(REPO)

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

PAIR = REPO / "data" / "custom" / "my_pair"
body_path, face_path = PAIR / "body.png", PAIR / "face.png"
if not (body_path.exists() and face_path.exists()):
    raise SystemExit("Images missing - run Cell 2 first.")

body_im = Image.open(body_path).convert("RGB")
face_im = Image.open(face_path).convert("RGB")

runtime = get_shared_krea2_runtime(init_custom_nodes=True)
cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({
    "seed": int(SEED),
    "save_debug": False,
    "verbose": False,
    # Pre-step under test (default OFF in the yaml): edit the DONOR's
    # expression to match the TARGET's measured expression before T4's own
    # main pass + face_refine run. See docs/PIPELINE_STATE.md CHECKPOINT-13.
    "pre_edit_donor_expression": bool(RUN_DONOR_PRE_EDIT),
    # The lever under test. Bundles are cached per (unet, clip, vae, lora,
    # STRENGTH), so each sweep value loads its own bundle -- no stale reuse.
    "identity_lora_strength": float(IDENTITY_LORA_STRENGTH),
})
print(f"identity_lora_strength={IDENTITY_LORA_STRENGTH}  (T4 default 1.0)")
if RUN_DONOR_PRE_EDIT and not USE_CODE_DEFAULTS:
    cfg.update({
        "pre_edit_donor_expression_denoise": float(PRE_EDIT_DENOISE),
        "pre_edit_donor_expression_ref_boost": float(PRE_EDIT_REF_BOOST),
        "pre_edit_donor_expression_cfg": float(PRE_EDIT_CFG),
        "pre_edit_donor_expression_disable_lora": bool(PRE_EDIT_DISABLE_LORA),
    })
    print("knobs: hand-tuned from the Cell 3 form (USE_CODE_DEFAULTS=False)")
else:
    print("knobs: using code defaults synced by Cell 1")

OUT_DIR = REPO / "results" / "pre_edit_expression_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
result = create_pipeline(cfg, runtime=runtime).run(body_im, face_im, out_dir=OUT_DIR)
elapsed = time.perf_counter() - t0

meta = result.meta or {}
pre_edit = meta.get("pre_edit_donor_expression") or {}
route = meta.get("body_route") or {}
edit_mode = meta.get("edit_mode")
route_name = route.get("route")
print(f"\n{elapsed:.0f}s  mode={edit_mode}  route={route_name}  out={result.image.size}")
print(f"pre_edit_donor_expression: applied={pre_edit.get('applied')} reason={pre_edit.get('reason')}")
if pre_edit.get("target_expression"):
    te = pre_edit["target_expression"]
    print(f"  target expression measured: {te.get('label')} "
          f"(smile_ratio={te.get('smile_ratio')} open_ratio={te.get('open_ratio')})")
    print(f"  donor edit knobs: denoise={pre_edit.get('denoise')} "
          f"ref_boost={pre_edit.get('ref_boost')} cfg={pre_edit.get('cfg')} "
          f"steps={pre_edit.get('steps')} "
          f"identity_lora_disabled={pre_edit.get('identity_lora_disabled')}")

pre_edit_face_path = OUT_DIR / "debug_pre_edit_donor_face.png"

display(Markdown("### 1 - Donor face (uploaded)"))
display(face_im)

if pre_edit.get("applied") and pre_edit_face_path.is_file():
    display(Markdown(
        "### 2 - Donor after the expression edit "
        "(pure Krea2 generation, no mask -- BEFORE T4's two-step pass)"
    ))
    display(Image.open(pre_edit_face_path))
else:
    skip_reason = pre_edit.get("reason")
    display(Markdown(
        f"### 2 - Donor expression edit SKIPPED ({skip_reason}) "
        "-- T4 ran on the original donor face"
    ))

display(Markdown(
    "### 3 - Final result (T4's two-step pass -- main pass + face_refine "
    "-- using the edited donor above)"
))
display(result.image)

final_path = OUT_DIR / "final_result.png"
result.image.save(final_path)
print(f"\nSaved: {final_path}")
if pre_edit_face_path.is_file():
    print(f"Saved: {pre_edit_face_path}")

# Uncomment to download:
# from google.colab import files; files.download(str(final_path))


In [ ]:
#@title Cell 4 - Expression-transfer PROBE (one Krea2 sample, no T4)
# Inverts the one mechanism this model does reliably. CHECKPOINTs 11-14 spent
# four sessions trying to STOP image 2's expression from riding along with its
# identity, across seven levers, and never once succeeded. So stop fighting it
# and point it the other way:
#
#     image 1 (scene)  = the DONOR   -> the identity we want to KEEP
#     image 2 (person) = the TARGET  -> the expression we want to TAKE
#
# If it works, the output is the donor wearing the target's expression, which
# is exactly the donor image T4 wants as input.
#
# The roles are swapped IN CODE. Upload normally in Cell 2 (body = target,
# face = donor) -- do not swap the uploads by hand.
#
# Identity LoRA defaults OFF: transplanting a whole head from image 2 is what
# it is trained to do, and here that is the failure mode, not the goal.
# THE EXPRESSION YOU WANT, in plain words, stated as a FACT about the
# picture being made. Round 1 of this probe asked the model to "match the
# expression of the person in the second image" -- a meta-instruction about
# which input to obey, which CHECKPOINT-11 already measured as inert twice
# and which was inert here too (identity held perfectly, expression did not
# move). Stating the expression as fact is the only phrasing shape with a
# working precedent in this repo, and it also sidesteps the broken openness
# measurement (CHECKPOINT-14) -- you know the expression you want.
#
# Leave blank to fall back to the old image-2 phrasing (not recommended).
PROBE_EXPRESSION = "not smiling, with a closed mouth and a neutral, serious expression"  #@param {type:"string"}

# Round 1 held identity but moved nothing. These are the "make the transfer
# actually happen" values: more image-2 influence, more room to move, and
# the LoRA back on -- it is possible the image2 -> image1 transfer we could
# never suppress IS the LoRA, in which case turning it off removed the very
# mechanism this probe is trying to exploit.
PROBE_LORA_STRENGTH = 0.5  #@param {type:"number"}
PROBE_DENOISE = 0.6  #@param {type:"number"}
PROBE_REF_BOOST = 3.5  #@param {type:"number"}
PROBE_CFG = 4.0  #@param {type:"number"}
SEED = 46  #@param {type:"integer"}

import sys, os
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
os.chdir(REPO)

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

PAIR = REPO / "data" / "custom" / "my_pair"
body_path, face_path = PAIR / "body.png", PAIR / "face.png"
if not (body_path.exists() and face_path.exists()):
    raise SystemExit("Images missing - run Cell 2 first.")

target_im = Image.open(body_path).convert("RGB")   # expression source
donor_im = Image.open(face_path).convert("RGB")    # identity to keep

runtime = get_shared_krea2_runtime(init_custom_nodes=True)
cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({
    "seed": int(SEED),
    "verbose": False,
    "probe_expression_text": str(PROBE_EXPRESSION).strip(),
    "probe_expression_lora_strength": float(PROBE_LORA_STRENGTH),
    "probe_expression_denoise": float(PROBE_DENOISE),
    "probe_expression_ref_boost": float(PROBE_REF_BOOST),
    "probe_expression_cfg": float(PROBE_CFG),
})

OUT_DIR = REPO / "results" / "expr_probe"
pipe = create_pipeline(cfg, runtime=runtime)
res = pipe.probe_expression_transfer(
    identity_img=donor_im,       # image 1 -> keep this face
    expression_img=target_im,    # image 2 -> take this expression
    out_dir=OUT_DIR,
)

print(f"\n{res['latency_s']}s  lora_strength={res['lora_strength']} "
      f"denoise={res['denoise']} ref_boost={res['ref_boost']} cfg={res['cfg']}")
print(f"prompt: {res['prompt']}")

display(Markdown("### 1 - IDENTITY to keep (donor, = image 1 / scene)"))
display(donor_im)
display(Markdown("### 2 - EXPRESSION to take (target, = image 2 / person)"))
display(target_im)
display(Markdown("### 3 - PROBE OUTPUT"))
display(res["image"])

display(Markdown(
    "**Judge two things, they are separate questions:**\n\n"
    "1. Did the **expression** move toward image 2?\n"
    "2. Is it still **image 1's person**?\n\n"
    "Both yes -> this is the expression step; feed the output to T4 as the donor.\n\n"
    "Expression moved but it is now image 2's face -> the channel carries "
    "identity too; raising cfg / lowering ref_boost is the next thing to try.\n\n"
    "Nothing moved -> the transfer only works through the LoRA, and this "
    "route is closed like the others."
))


In [ ]:
#@title Cell 5 - LivePortrait: change ONLY the expression (installs on first run)
# Purpose-built expression transfer, driven by the target photo's own dense
# keypoints -- no text in the loop, which is the bottleneck that killed every
# Krea2 attempt (CHECKPOINT-11..14: eight levers, five sessions, no movement).
#
#   SOURCE  = the face whose identity we KEEP    (donor, or a T4 output)
#   DRIVING = the photo whose expression we TAKE (the target)
#
# The real logic lives in src/headswap/expression_transfer.py, which Cell 1
# syncs from git. That is deliberate: notebook #@param values are cached in
# the BROWSER TAB and survive a git sync, and stale values silently burned
# ~6 runs of this investigation. Anything correctness-critical (notably
# relative motion, which is a no-op for a single still driver) is enforced
# there, not here.
#
# LICENSE: LivePortrait code + weights are MIT. It uses InsightFace
# buffalo_l (non-commercial research only) -- but headswap_V2 already
# depends on buffalo_l, so this adds no new restriction.

SOURCE_IS = "donor"  #@param ["donor", "t4_output"]
# "lip" = mouth only, eye region untouched. "exp" is a fuller expression but
# can move the eyes.
ANIMATION_REGION = "lip"  #@param ["lip", "exp", "eyes", "pose", "all"]
# Raise to 1.2-1.5 if the change is real but too subtle; below 1.0 if the
# mouth overshoots into a grimace.
DRIVING_MULTIPLIER = 1.0  #@param {type:"number"}
STITCHING = True  #@param {type:"boolean"}

import os, sys, subprocess
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

LP = Path("/content/LivePortrait")
REPO = Path("/content/headswap_V2")

if not (LP / "inference.py").exists():
    print("-> cloning LivePortrait ...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KwaiVGI/LivePortrait", str(LP)], check=True)
    # NEVER `pip install -r requirements.txt`: it pins torch/numpy and would
    # wreck the Krea2 env in this same runtime -- the exact failure this repo
    # already hit when simple-lama silently downgraded pillow/numpy and killed
    # rembg + restore_background with no error.
    print("-> installing curated deps (NOT requirements.txt) ...")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir",
                    "tyro", "imageio", "imageio-ffmpeg", "rich", "pykalman",
                    "ffmpeg-python"], check=False)
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                    "--no-deps", "numpy==2.4.6"], check=False)
else:
    print("LivePortrait already cloned - skipping install")

WEIGHTS = LP / "pretrained_weights"
if not (WEIGHTS / "liveportrait").exists():
    print("-> downloading weights (~660MB, humans only) ...")
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    from huggingface_hub import snapshot_download
    _err = None
    for _repo in ("KwaiVGI/LivePortrait", "KlingTeam/LivePortrait"):
        try:
            snapshot_download(repo_id=_repo, local_dir=str(WEIGHTS),
                              ignore_patterns=["*animal*"])
            print(f"   weights from {_repo}"); _err = None; break
        except Exception as exc:  # noqa: BLE001
            _err = exc; print(f"   {_repo} failed ({exc}); trying next")
    if _err is not None:
        raise SystemExit(f"weights download failed: {_err}")
else:
    print("weights already present - skipping download")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]
from headswap.expression_transfer import run_expression_transfer

PAIR = REPO / "data" / "custom" / "my_pair"
driving_path = PAIR / "body.png"
source_path = (PAIR / "face.png") if SOURCE_IS == "donor" else (
    REPO / "results" / "pre_edit_expression_test" / "final_result.png")
if not source_path.exists():
    raise SystemExit(f"missing {source_path} - run Cell 2 (and Cell 3 for t4_output) first.")

OUT = Path("/content/lp_out")
for old in OUT.glob("*"):
    if old.is_file():
        old.unlink()

res = run_expression_transfer(
    source_path=source_path,        # identity to KEEP
    driving_path=driving_path,      # expression to TAKE
    out_dir=OUT,
    animation_region=ANIMATION_REGION,
    driving_multiplier=float(DRIVING_MULTIPLIER),
    stitching=bool(STITCHING),
    live_portrait_dir=LP,
)
print(f"\n{res['latency_s']}s  region={res['animation_region']} "
      f"relative={res['relative_motion']} ({res['relative_motion_reason']}) "
      f"mult={res['driving_multiplier']}")

display(Markdown("### 1 - SOURCE (identity kept)"))
display(Image.open(source_path))
display(Markdown("### 2 - DRIVING (expression taken)"))
display(Image.open(driving_path))

if res["primary"]:
    out_img = Path(res["primary"])
    display(Markdown(f"### 3 - LIVEPORTRAIT OUTPUT (`{out_img.name}`)"))
    if out_img.suffix.lower() == ".mp4":
        import imageio.v3 as iio
        im = Image.fromarray(iio.imread(out_img, index=0))
    else:
        im = Image.open(out_img)
    display(im)
    final = REPO / "results" / "liveportrait_result.png"
    final.parent.mkdir(parents=True, exist_ok=True)
    im.convert("RGB").save(final)
    print(f"saved -> {final}")
else:
    print("No output produced:", res["produced"])

display(Markdown(
    "**Judge:** did the mouth move toward image 2, while the identity AND "
    "the eye region stayed as image 1?\n\n"
    "- **nothing moved** -> check `relative=` above; it must be False for a "
    "still driver (now enforced in code)\n"
    "- moved but too subtle -> DRIVING_MULTIPLIER 1.2-1.5\n"
    "- mouth overshoots / grimaces -> DRIVING_MULTIPLIER below 1.0\n"
    "- eyes moved too -> ANIMATION_REGION must be 'lip', not 'exp'/'all'\n"
    "- visible seam around the face -> turn STITCHING back on"
))


In [ ]:
#@title Cell 6 - Expression edit DENOISE SWEEP -> then the T4 swap
# NO numeric #@param fields, on purpose. Colab caches form values in the
# BROWSER TAB and `git reset --hard` cannot touch them, so a stale tab keeps
# re-sending old numbers -- that has silently confounded ~9 runs of this
# investigation (denoise stuck at 0.35, then at 1.0, while newly-added
# params picked up fresh code defaults). Every knob below is read from
# DENOISE_SWEEP in this cell's synced source, so what you see is what ran.
#
# Sweeps the expression edit at several denoise values in one go (~20s each)
# and prints ArcFace identity against the original donor for each, then runs
# the T4 swap once on the arm you pick. denoise=1.0 regenerates the donor
# completely; lower values let the source latent anchor the face.
#
# Upstream's own README: "structure-preserving i2i -- can't guarantee 1:1
# content preservation, confirmed not solved by us or the wider community."
# So expect a ceiling here; the goal is the best point on the curve, not 1.0.

EXPRESSION_INSTRUCTION = "change this person's facial expression so they are not smiling, with a closed relaxed mouth and a neutral, serious expression. Keep their eyes, identity, hair and pose exactly the same."  #@param {type:"string"}

DENOISE_SWEEP = [1.0, 0.8, 0.65, 0.5]   # code-owned, cannot go stale
SWAP_WITH = "best"   # "best" = highest identity that still changed expression, or a float like 0.65
SEED = 46

import sys, os, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
os.chdir(REPO)

import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime
from headswap.metrics.scoring import identity_cosine

PAIR = REPO / "data" / "custom" / "my_pair"
body_path, face_path = PAIR / "body.png", PAIR / "face.png"
if not (body_path.exists() and face_path.exists()):
    raise SystemExit("Images missing - run Cell 2 first.")
body_im = Image.open(body_path).convert("RGB")
face_im = Image.open(face_path).convert("RGB")

runtime = get_shared_krea2_runtime(init_custom_nodes=True)
cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
cfg.update({"seed": int(SEED), "verbose": False,
            "pre_edit_donor_expression": False})
pipe = create_pipeline(cfg, runtime=runtime)

OUT = REPO / "results" / "single_edit_test"
OUT.mkdir(parents=True, exist_ok=True)

display(Markdown("### Donor BEFORE"))
display(face_im)

# ---- Phase A: sweep denoise on the expression edit only -------------------
arms = []
for dn in DENOISE_SWEEP:
    pipe.cfg["single_edit_denoise"] = float(dn)
    r = pipe.edit_single_image(face_im, EXPRESSION_INSTRUCTION)
    idc = identity_cosine(face_im, r["image"])
    arms.append({"denoise": dn, "image": r["image"], "identity": idc,
                 "latency_s": r["latency_s"]})
    r["image"].save(OUT / f"edit_denoise_{dn}.png")
    print(f"  denoise={dn}  identity_vs_donor={idc}  ({r['latency_s']}s)")

display(Markdown("### Expression edit at each denoise"))
for a in arms:
    display(Markdown(f"**denoise={a['denoise']} · identity={a['identity']}**"))
    display(a["image"])

display(Markdown(
    "| denoise | identity vs donor |\n|---|---|\n" +
    "\n".join(f"| {a['denoise']} | {a['identity']} |" for a in arms)
))
print("\nPick the HIGHEST denoise whose expression actually changed while "
      "identity stays high -- too low and the expression stops moving, too "
      "high and the donor stops being the donor.")

# ---- Phase B: one T4 swap on the chosen arm -------------------------------
if SWAP_WITH == "best":
    scored = [a for a in arms if a["identity"] is not None]
    chosen = max(scored, key=lambda a: a["identity"]) if scored else arms[-1]
else:
    chosen = min(arms, key=lambda a: abs(a["denoise"] - float(SWAP_WITH)))

print(f"\nswapping with denoise={chosen['denoise']} "
      f"(identity={chosen['identity']})")
t0 = time.perf_counter()
result = pipe.run(body_im, chosen["image"], out_dir=OUT)
print(f"T4 swap: {time.perf_counter()-t0:.0f}s")

display(Markdown(f"### FINAL SWAP (edit denoise={chosen['denoise']})"))
display(result.image)
print(f"id_donor_vs_final  = {identity_cosine(face_im, result.image)}")
print(f"id_edited_vs_final = {identity_cosine(chosen['image'], result.image)}")
result.image.save(OUT / "final_result.png")
print(f"saved -> {OUT / 'final_result.png'}")
